# Phase 5: Statistical Modeling & Causal Proof
**Author:** Oyinlayefa Mezeh  
**Date:** March 2026  
**Objective:** This notebook applies econometric modeling (OLS Regression) and Time-Lag feature engineering to the `nigeria_analytical_master` dataset. The goal is to mathematically test the research hypothesis: measuring the statistically significant impact of environmental triggers (Drought) on market shocks (Food Prices), and subsequent societal unrest (Conflict Frequency).

## 1. Environment Setup & Time-Lag Engineering
We load the master dataset and create temporal lags. Since maize and rice take several months to harvest, we hypothesize that a climate shock (Drought) in Month $T$ will impact Food Prices in Month $T+3$ or $T+4$. We will shift our independent variables to account for this agricultural delay.

In [4]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

# 1. Load the Master Dataset
df = pd.read_csv('nigeria_analytical_master.csv')

# Ensure date is a datetime object and sort chronologically by state
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(by=['adm1_name', 'date']).reset_index(drop=True)

# 2. Time-Lag Feature Engineering
# We create 3-month and 4-month lags for Drought to see delayed impacts on Price
df['Drought_Lag3'] = df.groupby('adm1_name')['Drought_Active'].shift(3)
df['Drought_Lag4'] = df.groupby('adm1_name')['Drought_Active'].shift(4)

# We create a 1-month lag for Price to see its immediate impact on Conflict
df['Price_Norm_Lag1'] = df.groupby('adm1_name')['Price_Norm'].shift(1)

# Drop the NaN rows created by the lagging process so our regression model doesn't crash
df_model = df.dropna(subset=['Drought_Lag3', 'Drought_Lag4', 'Price_Norm_Lag1']).copy()

print(f"✅ Lag variables created. Model dataset ready with {len(df_model)} observations.")

✅ Lag variables created. Model dataset ready with 8213 observations.


## 2. Regression Model 1: Climate Impact on Food Prices
**Hypothesis:** A severe drought event significantly increases normalized staple food prices 3 to 4 months post-onset.
**Dependent Variable (Y):** `Price_Norm`
**Independent Variables (X):** `Drought_Lag3`, `Drought_Lag4`

In [2]:
# Define the Dependent Variable (Y) and Independent Variables (X)
Y1 = df_model['Price_Norm']
X1 = df_model[['Drought_Lag3', 'Drought_Lag4']]

# Add a constant (intercept) to the model - standard econometric practice
X1 = sm.add_constant(X1)

# Fit the OLS Regression Model
model_1 = sm.OLS(Y1, X1).fit()

# Print the Academic Summary
print("==============================================================================")
print("                   MODEL 1: IMPACT OF DROUGHT ON FOOD PRICES                  ")
print("==============================================================================")
print(model_1.summary())

                   MODEL 1: IMPACT OF DROUGHT ON FOOD PRICES                  
                            OLS Regression Results                            
Dep. Variable:             Price_Norm   R-squared:                       0.017
Model:                            OLS   Adj. R-squared:                  0.017
Method:                 Least Squares   F-statistic:                     70.42
Date:                Mon, 23 Mar 2026   Prob (F-statistic):           4.76e-31
Time:                        15:03:44   Log-Likelihood:                -59008.
No. Observations:                8213   AIC:                         1.180e+05
Df Residuals:                    8210   BIC:                         1.180e+05
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------

## 3. Regression Model 2: Price Volatility as a Threat Multiplier
**Hypothesis:** Spikes in staple food prices (lagged by 1 month to account for household exhaustion of reserves) significantly predict an increase in localized conflict frequency.
**Dependent Variable (Y):** `conflict_frequency`
**Independent Variables (X):** `Price_Norm_Lag1`

In [3]:
# Define the Dependent Variable (Y) and Independent Variable (X)
Y2 = df_model['conflict_frequency']
X2 = df_model[['Price_Norm_Lag1']]

# Add constant
X2 = sm.add_constant(X2)

# Fit the OLS Regression Model
model_2 = sm.OLS(Y2, X2).fit()

# Print the Academic Summary
print("==============================================================================")
print("                 MODEL 2: IMPACT OF FOOD PRICES ON CONFLICT                   ")
print("==============================================================================")
print(model_2.summary())

                 MODEL 2: IMPACT OF FOOD PRICES ON CONFLICT                   
                            OLS Regression Results                            
Dep. Variable:     conflict_frequency   R-squared:                       0.017
Model:                            OLS   Adj. R-squared:                  0.017
Method:                 Least Squares   F-statistic:                     142.9
Date:                Mon, 23 Mar 2026   Prob (F-statistic):           1.17e-32
Time:                        15:03:44   Log-Likelihood:                -29860.
No. Observations:                8213   AIC:                         5.972e+04
Df Residuals:                    8211   BIC:                         5.974e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------